# Module 18 Lab: LLM Gateway & Enterprise Infrastructure

**Mục tiêu:** Xây dựng LLM Gateway production-grade cho LoanBot — load balancing, failover, quota management, priority queue, SLA monitoring.

**Scenario:** FinTech Corp cần infrastructure để xử lý 50,000 hồ sơ/tháng với P99 latency ≤ 5s và uptime 99.9%.

---
**Sections:**
1. Gateway Core — Key pool, round-robin, cache
2. Quota Manager — Sliding window per team
3. Health Check System — Proactive detection
4. Priority Queue — P1/P2/P3 with intelligent routing
5. SLA Monitor — P99, error rate, cost attribution
6. Integration Test — Full pipeline simulation
7. Practice — Build backpressure mechanism

## Section 1: Gateway Core

Implement round-robin load balancing với exponential backoff retry và caching.

In [ ]:
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Callable
from collections import defaultdict, deque
from enum import Enum
import time
import hashlib
import json
import random

# === MOCK LLM CALL (không cần API key thật) ===
class MockApiError(Exception):
    pass

def mock_claude_call(api_key: str, model: str, messages: list, system: str = "") -> str:
    """Simulate Claude API call với random latency và occasional failures."""
    # Simulate key-specific failure rates
    key_health = {
        "key-A": {"fail_rate": 0.05, "latency_ms": 800},   # healthy
        "key-B": {"fail_rate": 0.30, "latency_ms": 2000},  # degraded
        "key-C": {"fail_rate": 0.05, "latency_ms": 900},   # healthy
        "backup-1": {"fail_rate": 0.02, "latency_ms": 1200}, # reliable backup
        "backup-2": {"fail_rate": 0.02, "latency_ms": 1100}, # reliable backup
    }
    health = key_health.get(api_key, {"fail_rate": 0.1, "latency_ms": 1000})
    
    # Simulate latency
    latency = health["latency_ms"] * random.uniform(0.8, 1.5) / 1000
    time.sleep(min(latency, 0.05))  # cap at 50ms for lab speed
    
    if random.random() < health["fail_rate"]:
        raise MockApiError(f"API key {api_key} returned 429 or 500")
    
    # Generate mock response based on content
    user_msg = messages[-1].get("content", "") if messages else ""
    if "blacklisted" in user_msg.lower():
        return json.dumps({"decision": "REJECT", "reason": "Customer on blacklist", "confidence": 0.99})
    elif "score" in user_msg and "720" in user_msg:
        return json.dumps({"decision": "APPROVE", "reason": "Strong credit profile", "confidence": 0.94})
    else:
        return json.dumps({"decision": "REVIEW", "reason": "Borderline case", "confidence": 0.72})

print("✅ Mock API setup complete")
print(f"  key-A: healthy (5% fail, 800ms)")
print(f"  key-B: degraded (30% fail, 2000ms)")
print(f"  key-C: healthy (5% fail, 900ms)")
print(f"  backup-1, backup-2: very healthy (2% fail)")

In [ ]:
@dataclass
class GatewayConfig:
    primary_keys:   List[str]
    backup_keys:    List[str]
    cache_ttl_s:    int   = 3600
    max_retries:    int   = 3
    rate_limit_rpm: int   = 60

class LLMGateway:
    def __init__(self, config: GatewayConfig, api_fn: Callable = mock_claude_call):
        self.config      = config
        self.api_fn      = api_fn
        self._key_idx    = 0
        self._cache: Dict[str, dict] = {}
        self._unhealthy_keys: set = set()
        self.metrics = defaultdict(int)  # requests, cache_hits, failures, pool_switches

    def _next_healthy_key(self, pool: List[str]) -> Optional[str]:
        """Round-robin nhưng skip unhealthy keys."""
        healthy = [k for k in pool if k not in self._unhealthy_keys]
        if not healthy:
            return None
        key = healthy[self._key_idx % len(healthy)]
        self._key_idx += 1
        return key

    def _cache_key(self, model: str, messages: list, system: str) -> str:
        payload = json.dumps({"model": model, "messages": messages, "system": system}, sort_keys=True)
        return hashlib.md5(payload.encode()).hexdigest()

    def mark_unhealthy(self, api_key: str):
        self._unhealthy_keys.add(api_key)
        print(f"  ⚠️  Key {api_key} marked unhealthy")

    def complete(self, model: str, messages: list, system: str = "", **kwargs) -> dict:
        """Main entry: cache → primary pool → backup pool → degraded."""
        self.metrics["requests"] += 1
        
        # 1. Cache check
        ck = self._cache_key(model, messages, system)
        if ck in self._cache:
            entry = self._cache[ck]
            if time.time() - entry["ts"] < self.config.cache_ttl_s:
                self.metrics["cache_hits"] += 1
                return {"response": entry["response"], "source": "cache", "api_key": "cache"}

        # 2. Try pools in order
        for pool, pool_name in [(self.config.primary_keys, "primary"),
                                 (self.config.backup_keys,  "backup")]:
            if not pool:
                continue
            if pool_name == "backup":
                self.metrics["pool_switches"] += 1

            for attempt in range(self.config.max_retries):
                api_key = self._next_healthy_key(pool)
                if api_key is None:
                    break  # all keys in pool unhealthy
                try:
                    response = self.api_fn(api_key, model, messages, system)
                    self._cache[ck] = {"ts": time.time(), "response": response}
                    return {"response": response, "source": pool_name, "api_key": api_key}
                except Exception as e:
                    sleep_s = min((2 ** attempt) * 0.01, 0.1)  # lab: ms range
                    time.sleep(sleep_s)
                    if attempt == self.config.max_retries - 1:
                        break  # try next pool

        # 3. All pools exhausted
        self.metrics["failures"] += 1
        raise RuntimeError("Gateway: all pools exhausted — degraded mode")

# Test
config = GatewayConfig(
    primary_keys=["key-A", "key-B", "key-C"],
    backup_keys=["backup-1", "backup-2"]
)
gateway = LLMGateway(config)

print("Testing gateway with 10 requests...")
sources = defaultdict(int)
for i in range(10):
    msgs = [{"role": "user", "content": f"Evaluate loan application FC-{i:04d} credit score 720"}]
    try:
        result = gateway.complete("claude-haiku-4-5", msgs)
        sources[result["source"]] += 1
    except RuntimeError:
        sources["failed"] += 1

print(f"\nMetrics: {dict(gateway.metrics)}")
print(f"Source distribution: {dict(sources)}")

## Section 2: Quota Manager

Sliding window quota per team để prevent any single department từ monopolizing API capacity.

In [ ]:
class QuotaManager:
    """Sliding window quota per team. window_s = 3600 → hourly limit."""
    
    def __init__(self, limits: Dict[str, int], window_s: int = 3600):
        self.limits   = limits
        self.window_s = window_s
        self.windows: Dict[str, deque] = defaultdict(deque)  # team → timestamps

    def _clean_window(self, team_id: str):
        """Remove timestamps outside the sliding window."""
        now    = time.time()
        window = self.windows[team_id]
        while window and now - window[0] > self.window_s:
            window.popleft()

    def can_request(self, team_id: str) -> bool:
        self._clean_window(team_id)
        limit = self.limits.get(team_id, 100)  # default 100/hour
        return len(self.windows[team_id]) < limit

    def record_request(self, team_id: str):
        self.windows[team_id].append(time.time())

    def usage_pct(self, team_id: str) -> float:
        self._clean_window(team_id)
        limit = self.limits.get(team_id, 100)
        return len(self.windows[team_id]) / limit * 100 if limit > 0 else 0

    def report(self) -> dict:
        return {
            team: {
                "used": len(self.windows[team]),
                "limit": self.limits.get(team, 100),
                "usage_pct": round(self.usage_pct(team), 1)
            }
            for team in self.limits
        }

# FinTech Corp quota config (per hour)
QUOTA_LIMITS = {
    "loan_dept":      250,   # primary business
    "risk_dept":       80,
    "compliance":      30,
    "test_env":        10,   # strict limit on test
    "priority_vip":    50,   # VIP fast lane
}

quota = QuotaManager(QUOTA_LIMITS)

# Simulate usage
print("Simulating department usage...")
for _ in range(8):
    quota.record_request("loan_dept")
for _ in range(15):
    quota.record_request("test_env")

print(f"\nTest env can_request: {quota.can_request('test_env')}")
print(f"(Test env limit: 10, used: 15 — should be BLOCKED)")
print(f"\nQuota report:")
for team, stats in quota.report().items():
    bar = '█' * int(stats['usage_pct'] / 10) + '░' * (10 - int(stats['usage_pct'] / 10))
    print(f"  {team:20s} [{bar}] {stats['used']:3d}/{stats['limit']:3d} ({stats['usage_pct']}%)")

## Section 3: Health Check System

Proactive health monitoring — phát hiện degraded keys trước khi ảnh hưởng production.

In [ ]:
@dataclass
class KeyHealth:
    api_key:      str
    status:       str = "healthy"      # healthy | degraded | unhealthy
    error_rate:   float = 0.0
    avg_latency:  float = 0.0
    last_checked: float = field(default_factory=time.time)
    check_count:  int = 0
    error_count:  int = 0

class HealthChecker:
    DEGRADED_LATENCY_MS  = 3000   # avg > 3s → degraded
    UNHEALTHY_ERROR_RATE = 0.10   # error rate > 10% → unhealthy
    COOLDOWN_S           = 30     # re-check after 30s (lab: shorter than prod 300s)

    def __init__(self, all_keys: List[str], api_fn: Callable = mock_claude_call):
        self.api_fn  = api_fn
        self.health  = {k: KeyHealth(api_key=k) for k in all_keys}

    def check_key(self, api_key: str) -> KeyHealth:
        """Run health check probe on a single key."""
        h = self.health[api_key]
        probe_msgs = [{"role": "user", "content": "health check probe"}]
        
        start = time.time()
        try:
            self.api_fn(api_key, "claude-haiku-4-5", probe_msgs)
            latency_ms = (time.time() - start) * 1000
            # Update rolling avg
            h.avg_latency = (h.avg_latency * h.check_count + latency_ms) / (h.check_count + 1)
        except:
            h.error_count += 1
        
        h.check_count  += 1
        h.error_rate    = h.error_count / h.check_count
        h.last_checked  = time.time()

        # Classify status
        if h.error_rate >= self.UNHEALTHY_ERROR_RATE:
            h.status = "unhealthy"
        elif h.avg_latency >= self.DEGRADED_LATENCY_MS:
            h.status = "degraded"
        else:
            h.status = "healthy"
        return h

    def check_all(self) -> dict:
        results = {}
        for key in self.health:
            h = self.check_key(key)
            results[key] = h
        return results

    def healthy_keys(self) -> List[str]:
        return [k for k, h in self.health.items() if h.status == "healthy"]

# Run health checks
checker = HealthChecker(["key-A", "key-B", "key-C", "backup-1", "backup-2"])

print("Running health checks (5 probes per key)...")
for _ in range(5):
    checker.check_all()

print("\nHealth Status Report:")
icons = {"healthy": "✅", "degraded": "⚠️ ", "unhealthy": "❌"}
for key, h in checker.health.items():
    icon = icons[h.status]
    print(f"  {icon} {key:12s} status={h.status:10s} error_rate={h.error_rate:.1%} avg_latency={h.avg_latency:.0f}ms")

print(f"\n✅ Healthy keys for routing: {checker.healthy_keys()}")

## Section 4: Priority Queue & Intelligent Routing

In [ ]:
import heapq
from dataclasses import dataclass, field

class Priority(Enum):
    P1_VIP    = 1
    P2_STANDARD = 2
    P3_BATCH  = 3

@dataclass(order=True)
class LoanRequest:
    priority:    int           # lower = higher priority (for min-heap)
    arrived_at:  float         # secondary sort: earlier = higher priority
    request_id:  str  = field(compare=False)
    team_id:     str  = field(compare=False)
    credit_score: int = field(compare=False)
    loan_amount_m: float = field(compare=False)  # millions VND
    blacklisted: bool = field(compare=False, default=False)

def assign_priority(credit_score: int, loan_amount_m: float, is_vip: bool, blacklisted: bool) -> Priority:
    """Business rule: assign request priority."""
    if is_vip or loan_amount_m >= 2000:  # VIP or large loan
        return Priority.P1_VIP
    elif blacklisted:                    # fast-reject (P1 for speed)
        return Priority.P1_VIP
    elif credit_score >= 600:            # standard processing
        return Priority.P2_STANDARD
    else:                               # borderline needs review queue
        return Priority.P3_BATCH

def select_model(credit_score: int, loan_amount_m: float, blacklisted: bool) -> str:
    """Intelligent routing: choose model based on complexity."""
    if blacklisted:
        return "claude-haiku-4-5"       # instant reject
    if credit_score >= 700 and loan_amount_m < 1000:
        return "claude-haiku-4-5"       # clear approval
    return "claude-sonnet-4-6"          # borderline or high-value

class PriorityGateway:
    """LLM Gateway với priority queue và intelligent routing."""
    
    def __init__(self, gateway: LLMGateway, quota: QuotaManager):
        self.gateway = gateway
        self.quota   = quota
        self._heap: List[LoanRequest] = []
        self.processed: List[dict] = []

    def enqueue(self, request: LoanRequest):
        heapq.heappush(self._heap, request)

    def process_next(self) -> Optional[dict]:
        if not self._heap:
            return None
        req = heapq.heappop(self._heap)
        
        # Quota check
        if not self.quota.can_request(req.team_id):
            return {"request_id": req.request_id, "status": "QUOTA_EXCEEDED", "queued_back": True}
        
        # Route to model
        model = select_model(req.credit_score, req.loan_amount_m, req.blacklisted)
        self.quota.record_request(req.team_id)
        
        msgs = [{"role": "user", "content": (
            f"Evaluate loan application {req.request_id}. "
            f"Credit score: {req.credit_score}. "
            f"Loan amount: {req.loan_amount_m}M VND. "
            f"{'BLACKLISTED CUSTOMER' if req.blacklisted else ''}"
        )}]
        
        start = time.time()
        try:
            result = self.gateway.complete(model, msgs)
            latency_ms = int((time.time() - start) * 1000)
            decision = json.loads(result["response"]).get("decision", "UNKNOWN")
            record = {
                "request_id": req.request_id, "priority": Priority(req.priority).name,
                "model": model, "decision": decision,
                "latency_ms": latency_ms, "source": result["source"]
            }
            self.processed.append(record)
            return record
        except RuntimeError as e:
            return {"request_id": req.request_id, "status": "GATEWAY_FAILURE", "error": str(e)}

# === Test with mixed batch ===
pg = PriorityGateway(gateway=LLMGateway(config), quota=QuotaManager(QUOTA_LIMITS))

# Simulate burst arrival (mixed priority)
test_batch = [
    # (id, team, score, amount, vip, blacklisted)
    ("FC-001", "loan_dept", 720, 500,    True,  False),  # P1 VIP
    ("FC-002", "loan_dept", 580, 300,    False, False),  # P2 Standard
    ("FC-003", "loan_dept", 400, 200,    False, True),   # P1 (blacklist fast-reject)
    ("FC-004", "loan_dept", 645, 800,    False, False),  # P2 Standard
    ("FC-005", "loan_dept", 750, 2500,   False, False),  # P1 (large loan)
    ("FC-006", "test_env",  600, 100,    False, False),  # P2 but test quota
    ("FC-007", "loan_dept", 480, 150,    False, False),  # P3 Batch
]

print("Enqueueing mixed batch...")
for rid, team, score, amount, vip, black in test_batch:
    priority = assign_priority(score, amount, vip, black)
    req = LoanRequest(
        priority=priority.value, arrived_at=time.time(),
        request_id=rid, team_id=team,
        credit_score=score, loan_amount_m=amount, blacklisted=black
    )
    pg.enqueue(req)
    print(f"  Enqueued {rid} → {priority.name} (score={score}, amount={amount}M)")

print("\nProcessing in priority order:")
while pg._heap:
    result = pg.process_next()
    if result:
        priority = result.get('priority', 'N/A')
        decision = result.get('decision', result.get('status', 'N/A'))
        model = result.get('model', 'N/A').replace('claude-', '').replace('-4-5','').replace('-4-6','')
        latency = result.get('latency_ms', 0)
        print(f"  [{priority:15s}] {result['request_id']} → {decision:8s} via {model:7s} ({latency}ms)")

## Section 5: SLA Monitor & Cost Attribution

In [ ]:
MODEL_COSTS = {
    "claude-haiku-4-5":  {"input": 0.25,  "output": 1.25},   # $/1M tokens
    "claude-sonnet-4-6": {"input": 3.00,  "output": 15.00},
}

AVG_TOKENS = {
    "claude-haiku-4-5":  {"input": 500,  "output": 200},   # typical LoanBot call
    "claude-sonnet-4-6": {"input": 800,  "output": 350},
}

@dataclass
class RequestLog:
    request_id:  str
    team_id:     str
    model:       str
    priority:    str
    latency_ms:  int
    success:     bool
    timestamp:   float = field(default_factory=time.time)
    
    @property
    def cost_usd(self) -> float:
        c   = MODEL_COSTS.get(self.model, {"input": 0, "output": 0})
        tok = AVG_TOKENS.get(self.model, {"input": 500, "output": 200})
        return (tok["input"] * c["input"] + tok["output"] * c["output"]) / 1_000_000

class SLAMonitor:
    SLA_P99_MS = 5000     # 5 second P99
    SLA_ERROR  = 0.001    # 0.1% error rate
    ALERT_P99  = 4000     # alert at 80% of SLA

    def __init__(self):
        self.logs: List[RequestLog] = []
        self.cost_by_team: Dict[str, float] = defaultdict(float)
        self.alerts: List[str] = []

    def record(self, log: RequestLog):
        self.logs.append(log)
        self.cost_by_team[log.team_id] += log.cost_usd
        self._check_alerts()

    def _check_alerts(self):
        if len(self.logs) < 10:  # need minimum sample
            return
        r = self.compute_report()
        if r["p99_latency_ms"] > self.ALERT_P99:
            alert = f"⚠️  SLA WARNING: P99={r['p99_latency_ms']}ms (threshold {self.SLA_P99_MS}ms)"
            if alert not in self.alerts:
                self.alerts.append(alert)
        if r["error_rate"] > self.SLA_ERROR * 0.5:
            alert = f"🚨 ERROR RATE: {r['error_rate']:.4%} approaching SLA limit"
            if alert not in self.alerts:
                self.alerts.append(alert)

    def compute_report(self) -> dict:
        if not self.logs:
            return {}
        latencies = sorted([l.latency_ms for l in self.logs])
        p99_idx   = max(0, int(len(latencies) * 0.99) - 1)
        errors    = sum(1 for l in self.logs if not l.success)
        total_cost = sum(l.cost_usd for l in self.logs)
        
        # Cost by model
        cost_by_model = defaultdict(float)
        for l in self.logs:
            cost_by_model[l.model] += l.cost_usd
        
        return {
            "total_requests":  len(self.logs),
            "p50_latency_ms":  latencies[len(latencies)//2],
            "p99_latency_ms":  latencies[p99_idx],
            "p99_sla_ok":      latencies[p99_idx] <= self.SLA_P99_MS,
            "error_rate":      errors / len(self.logs),
            "error_sla_ok":    errors / len(self.logs) <= self.SLA_ERROR,
            "total_cost_usd":  round(total_cost, 4),
            "cost_by_team":    {k: round(v, 4) for k, v in sorted(self.cost_by_team.items(), key=lambda x:-x[1])},
            "cost_by_model":   {k: round(v, 4) for k, v in dict(cost_by_model).items()},
        }

# Simulate a month of requests (compressed)
monitor = SLAMonitor()
teams = ["loan_dept", "risk_dept", "compliance", "test_env"]
team_weights = [0.65, 0.22, 0.09, 0.04]
models = ["claude-haiku-4-5", "claude-sonnet-4-6"]
model_weights = [0.60, 0.40]  # 60% haiku, 40% sonnet

random.seed(42)
for i in range(200):  # simulate 200 requests
    team  = random.choices(teams, weights=team_weights)[0]
    model = random.choices(models, weights=model_weights)[0]
    # Realistic latency distribution
    base_latency = 800 if "haiku" in model else 1400
    latency = int(random.lognormvariate(0, 0.5) * base_latency)
    success = random.random() > 0.003  # 0.3% error rate
    log = RequestLog(
        request_id=f"FC-{i:04d}", team_id=team, model=model,
        priority="P2_STANDARD", latency_ms=latency, success=success
    )
    monitor.record(log)

report = monitor.compute_report()
print("=" * 55)
print("  SLA DASHBOARD — LoanBot Monthly Report")
print("=" * 55)
print(f"  Total requests:    {report['total_requests']:,}")
print(f"  P50 latency:       {report['p50_latency_ms']:,}ms")
print(f"  P99 latency:       {report['p99_latency_ms']:,}ms  {'✅ OK' if report['p99_sla_ok'] else '❌ BREACH'}")
print(f"  Error rate:        {report['error_rate']:.4%}  {'✅ OK' if report['error_sla_ok'] else '❌ BREACH'}")
print(f"  Total cost:        ${report['total_cost_usd']:.4f}")
print()
print("  Cost Attribution:")
for team, cost in report['cost_by_team'].items():
    pct = cost / report['total_cost_usd'] * 100
    bar = '█' * int(pct / 5) + '░' * (20 - int(pct / 5))
    print(f"    {team:20s} ${cost:.4f}  ({pct:.1f}%) {bar}")
print()
print("  Cost by Model:")
for model, cost in report['cost_by_model'].items():
    model_short = model.replace('claude-', '').replace('-4-5','').replace('-4-6','')
    print(f"    {model_short:15s} ${cost:.4f}")

if monitor.alerts:
    print()
    print("  ALERTS:")
    for a in monitor.alerts:
        print(f"    {a}")

## Section 6: Integration Test — Full Pipeline

Kết hợp tất cả components vào một production-ready gateway system.

In [ ]:
class ProductionGateway:
    """Full production gateway: health check + quota + priority + SLA monitoring."""

    def __init__(self, config: GatewayConfig, quota_limits: dict):
        self.base_gateway = LLMGateway(config)
        self.quota        = QuotaManager(quota_limits)
        self.health       = HealthChecker(
            config.primary_keys + config.backup_keys,
            mock_claude_call
        )
        self.monitor      = SLAMonitor()
        self._queue: List[LoanRequest] = []

    def submit(self, request_id: str, team_id: str, credit_score: int,
               loan_amount_m: float, is_vip: bool = False, blacklisted: bool = False) -> str:
        """Submit request to priority queue."""
        priority = assign_priority(credit_score, loan_amount_m, is_vip, blacklisted)
        req = LoanRequest(
            priority=priority.value, arrived_at=time.time(),
            request_id=request_id, team_id=team_id,
            credit_score=credit_score, loan_amount_m=loan_amount_m, blacklisted=blacklisted
        )
        heapq.heappush(self._queue, req)
        return f"Queued as {priority.name} (queue depth: {len(self._queue)})"

    def process_batch(self, max_requests: int = 10) -> List[dict]:
        """Process up to max_requests from queue in priority order."""
        results = []
        processed = 0
        while self._queue and processed < max_requests:
            req = heapq.heappop(self._queue)

            # Quota gate
            if not self.quota.can_request(req.team_id):
                heapq.heappush(self._queue, req)  # put back
                results.append({"request_id": req.request_id, "status": "QUOTA_EXCEEDED"})
                continue

            # Route to model
            model = select_model(req.credit_score, req.loan_amount_m, req.blacklisted)
            self.quota.record_request(req.team_id)

            msgs = [{"role": "user", "content": (
                f"Loan {req.request_id}: score={req.credit_score} "
                f"amount={req.loan_amount_m}M "
                f"{'BLACKLISTED' if req.blacklisted else ''}"
            )}]

            start = time.time()
            success = True
            try:
                result = self.base_gateway.complete(model, msgs)
                decision = json.loads(result["response"]).get("decision", "UNKNOWN")
            except RuntimeError:
                decision = "GATEWAY_FAILURE"
                success = False

            latency_ms = int((time.time() - start) * 1000)
            log = RequestLog(
                request_id=req.request_id, team_id=req.team_id,
                model=model, priority=Priority(req.priority).name,
                latency_ms=latency_ms, success=success
            )
            self.monitor.record(log)
            results.append({"request_id": req.request_id, "priority": Priority(req.priority).name,
                            "decision": decision, "model": model.split('-')[1], "latency_ms": latency_ms})
            processed += 1
        return results

# === Integration test ===
prod_config = GatewayConfig(primary_keys=["key-A", "key-C"], backup_keys=["backup-1"])
prod_gw = ProductionGateway(prod_config, QUOTA_LIMITS)

# Submit mixed batch
print("Submitting requests...")
applications = [
    ("FC-2024-001", "loan_dept", 720, 500,  True,  False),
    ("FC-2024-002", "loan_dept", 580, 300,  False, False),
    ("FC-2024-003", "loan_dept", 400, 200,  False, True),
    ("FC-2024-004", "loan_dept", 645, 800,  False, False),
    ("FC-2024-005", "risk_dept", 700, 1500, False, False),
]
for rid, team, score, amount, vip, black in applications:
    status = prod_gw.submit(rid, team, score, amount, vip, black)
    print(f"  {rid}: {status}")

print("\nProcessing (priority order):")
results = prod_gw.process_batch(max_requests=5)
for r in results:
    print(f"  [{r.get('priority','N/A'):15s}] {r['request_id']} → {r.get('decision','N/A'):7s} ({r.get('latency_ms',0)}ms)")

print("\nSLA Report:")
rpt = prod_gw.monitor.compute_report()
if rpt:
    print(f"  P99 latency: {rpt.get('p99_latency_ms','N/A')}ms | Error rate: {rpt.get('error_rate',0):.4%}")
    print(f"  Total cost: ${rpt.get('total_cost_usd', 0):.5f}")
else:
    print("  (Insufficient data for SLA report)")

## Section 7: Practice — Implement Backpressure Mechanism

**Challenge:** Khi queue depth vượt ngưỡng, gateway phải từ chối requests mới để tránh cascade failure.

Implement hàm `BackpressureGateway` với:
1. `max_queue_size`: từ chối submissions khi queue đầy
2. `warn_threshold`: cảnh báo khi queue ở 80% capacity
3. `submit()` returns: `ACCEPTED`, `QUEUE_WARNING`, hoặc `REJECTED_BACKPRESSURE`

In [ ]:
# TODO: Implement BackpressureGateway
class BackpressureGateway:
    def __init__(self, base_gw: ProductionGateway, max_queue_size: int = 100, warn_threshold: float = 0.80):
        self.base_gw         = base_gw
        self.max_queue_size  = max_queue_size
        self.warn_threshold  = warn_threshold

    def submit(self, request_id: str, team_id: str, credit_score: int,
               loan_amount_m: float, is_vip: bool = False, blacklisted: bool = False) -> dict:
        """Submit with backpressure.
        Returns: {'status': 'ACCEPTED'|'QUEUE_WARNING'|'REJECTED_BACKPRESSURE', 'queue_depth': int}
        """
        # TODO: implement
        pass

# Test your implementation:
# bpgw = BackpressureGateway(ProductionGateway(prod_config, QUOTA_LIMITS), max_queue_size=5)
# for i in range(8):
#     result = bpgw.submit(f"FC-TEST-{i}", "loan_dept", 650, 300)
#     print(f"  Request {i}: {result}")

In [ ]:
# Solution (xem sau khi đã thử)
def show_solution():
    solution = '''
class BackpressureGateway:
    def __init__(self, base_gw, max_queue_size=100, warn_threshold=0.80):
        self.base_gw        = base_gw
        self.max_queue_size = max_queue_size
        self.warn_threshold = warn_threshold

    def submit(self, request_id, team_id, credit_score, loan_amount_m,
               is_vip=False, blacklisted=False):
        current_depth = len(self.base_gw._queue)
        
        # Hard limit — reject to protect system
        if current_depth >= self.max_queue_size:
            return {"status": "REJECTED_BACKPRESSURE", "queue_depth": current_depth,
                    "message": f"Queue full ({current_depth}/{self.max_queue_size}). Retry in 60s."}
        
        # Accept request
        status_msg = self.base_gw.submit(request_id, team_id, credit_score, loan_amount_m, is_vip, blacklisted)
        new_depth  = len(self.base_gw._queue)
        
        # Warn if approaching limit
        if new_depth >= int(self.max_queue_size * self.warn_threshold):
            return {"status": "QUEUE_WARNING", "queue_depth": new_depth,
                    "message": f"Queue at {new_depth/self.max_queue_size:.0%} capacity — consider slowing submissions"}
        
        return {"status": "ACCEPTED", "queue_depth": new_depth}
    '''
    print(solution)

# Uncomment to see solution:
# show_solution()

## Summary

Bạn đã implement đầy đủ **LLM Gateway & Enterprise Infrastructure**:

| Component | Key Concept |
|-----------|-------------|
| `LLMGateway` | Round-robin key pool + cache + exponential backoff failover |
| `QuotaManager` | Sliding window rate limiting per team |
| `HealthChecker` | Proactive key health monitoring |
| `PriorityGateway` | Min-heap priority queue + intelligent model routing |
| `SLAMonitor` | P99 latency, error rate, cost attribution |
| `ProductionGateway` | Integration: all components working together |
| `BackpressureGateway` | Queue depth protection (practice exercise) |

**LoanBot v3.4 key metrics:**
- VIP requests: ≤30s wait (P1 queue)
- Standard: ≤5 min (P2)
- Batch/reprocessing: best-effort (P3)
- Cost savings vs single-model: ~59% (mixed haiku/sonnet routing)
- 99.9% uptime via 3-level failover